<a href="https://colab.research.google.com/github/akshitmittal-official/nlp-chatbot/blob/main/notebooks/chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("Hello chatbot")

Hello chatbot


In [ ]:
import json

intents = {
    "intents": [
        {
            "tag": "greeting",
            "patterns": ["hi", "hello", "hey", "good morning"],
            "responses": ["Hello!", "Hi there!", "Hey!"]
        },
        {
            "tag": "goodbye",
            "patterns": ["bye", "goodbye", "see you"],
            "responses": ["Goodbye!", "See you later!"]
        },
        {
            "tag": "name",
            "patterns": ["what is your name", "who are you"],
            "responses": ["I am your NLP chatbot."]
        }
    ]
}

with open("intents.json", "w") as file:
    json.dump(intents, file, indent=4)

print("intents.json created successfully!")

intents.json created successfully!


In [ ]:
import json

with open("intents.json", "r") as file:
    data = json.load(file)

patterns = []
tags = []

for intent in data["intents"]:
    for pattern in intent["patterns"]:
        patterns.append(pattern)
        tags.append(intent["tag"])

print("Patterns:")
print(patterns)

print("\nTags:")
print(tags)

Patterns:
['hi', 'hello', 'hey', 'good morning', 'bye', 'goodbye', 'see you', 'ok bye', 'what is your name', 'who are you', 'what is python', 'what is a loop', 'how do loops work', 'what is a function']

Tags:
['greeting', 'greeting', 'greeting', 'greeting', 'goodbye', 'goodbye', 'goodbye', 'goodbye', 'name', 'name', 'python', 'loop', 'loop', 'function']


In [ ]:
# Understanding , in train_chatbot
clean_patterns = []

for sentence in patterns:
    words = nltk.word_tokenize(sentence.lower())
    clean_patterns.append(words)

print(clean_patterns)

NameError: name 'patterns' is not defined

In [ ]:
sentences = []

for words in clean_patterns:
    sentence = " ".join(words)
    sentences.append(sentence)

print(sentences)

['hi', 'hello', 'hey', 'good morning', 'bye', 'goodbye', 'see you', 'ok bye', 'what is your name', 'who are you', 'what is python', 'what is a loop', 'how do loops work', 'what is a function']


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()

X = vectorizer.fit_transform(sentences)

print(X.toarray())

NameError: name 'sentences' is not defined

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X, tags)

print("Model trained successfully!")

NameError: name 'X' is not defined

In [ ]:
# older response code , can't give proper answers if confused
import random

def chatbot_response(user_input):
    user_input = user_input.lower()

    user_vector = vectorizer.transform([user_input])

    prediction = model.predict(user_vector)[0]

    for intent in data["intents"]:
        if intent["tag"] == prediction:
            return random.choice(intent["responses"])

In [ ]:
# Could answer confidently but answers on the basis of data inside intents only
import random

def chatbot_response(user_input):
    user_input = user_input.lower()

    user_vector = vectorizer.transform([user_input])

    probabilities = model.predict_proba(user_vector)[0]

    max_probability = max(probabilities)

    if max_probability < 0.5:
        return "Sorry, I don't understand that yet."

    prediction = model.predict(user_vector)[0]

    for intent in data["intents"]:
        if intent["tag"] == prediction:
            return random.choice(intent["responses"])

In [ ]:
# now uses spaCy and WordNet for words and grammer
import random

def chatbot_response(user_input):
    user_input = user_input.lower()

    user_vector = vectorizer.transform([user_input])

    probabilities = model.predict_proba(user_vector)[0]
    max_probability = max(probabilities)

    if max_probability >= 0.5:
        prediction = model.predict(user_vector)[0]

        for intent in data["intents"]:
            if intent["tag"] == prediction:
                return random.choice(intent["responses"])

    doc = nlp(user_input)

    for token in doc:
        if token.pos_ == "NOUN":
            meanings = wordnet.synsets(token.text)

            if meanings:
                return f"{token.text}: {meanings[0].definition()}"

    return "I am not sure what that means."

In [41]:
# More improved
import random
from nltk.corpus import wordnet

def chatbot_response(user_input):
    user_input = user_input.lower().strip()

    # 1. Handle questions like: "what is python in nature"
    if user_input.startswith("what is ") and " in " in user_input:
        word = user_input.replace("what is ", "").strip()

        parts = word.split(" in ")

        main_word = parts[0].strip()
        context = parts[1].strip()

        # Remove "a" or "an"
        if main_word.startswith("a "):
            main_word = main_word[2:]

        if main_word.startswith("an "):
            main_word = main_word[3:]

        meanings = wordnet.synsets(main_word)

        if meanings:
            return f"In {context}, {main_word} means: {meanings[0].definition()}"

    # 2. Handle questions like: "what is python"
    if user_input.startswith("what is "):
        word = user_input.replace("what is ", "").strip()

        # Remove "a" or "an"
        if word.startswith("a "):
            word = word[2:]

        if word.startswith("an "):
            word = word[3:]

        # First check if the word belongs to one of your trained intents
        for intent in data["intents"]:
            for pattern in intent["patterns"]:
                if word in pattern:
                    return random.choice(intent["responses"])

        # Otherwise use WordNet
        meanings = wordnet.synsets(word)

        if meanings:
            return f"{word}: {meanings[0].definition()}"

    # 3. Use spaCy similarity for other sentence styles
    best_score = 0
    best_intent = None

    user_doc = nlp(user_input)

    for intent in data["intents"]:
        for pattern in intent["patterns"]:
            pattern_doc = nlp(pattern.lower())

            score = user_doc.similarity(pattern_doc)

            if score > best_score:
                best_score = score
                best_intent = intent

    if best_score > 0.65:
        return random.choice(best_intent["responses"])

    # 4. Final fallback
    return "I am not sure yet. Try asking in another way."

In [ ]:
# Understanding , in train_chatbot
while True:
    message = input("You: ")

    if message.lower() == "quit":
        print("Bot: Goodbye!")
        break

    response = chatbot_response(message)
    print("Bot:", response)

You: what is python ?
Bot: Hi there!
You: what is a loop
Bot: Hey!
You: quit
Bot: Goodbye!


In [32]:
intents = {
    "intents": [
        {
            "tag": "greeting",
            "patterns": ["hi", "hello", "hey", "good morning"],
            "responses": ["Hello!", "Hi there!", "Hey!"]
        },
        {
            "tag": "goodbye",
            "patterns": ["bye", "goodbye", "see you", "ok bye"],
            "responses": ["Goodbye!", "See you later!"]
        },
        {
            "tag": "name",
            "patterns": ["what is your name", "who are you"],
            "responses": ["I am your NLP chatbot."]
        },
        {
            "tag": "python",
            "patterns": ["what is python"],
            "responses": ["Python is a programming language."]
        },
        {
            "tag": "loop",
            "patterns": ["what is a loop", "how do loops work"],
            "responses": ["A loop repeats code."]
        },
        {
            "tag": "function",
            "patterns": ["what is a function"],
            "responses": ["A function is reusable code."]
        },
        {
          "tag": "variable",
          "patterns": ["what is a variable", "what are variables"],
          "responses": ["A variable stores data."]
        },
        {
          "tag": "list",
          "patterns": ["what is a list", "how do lists work"],
          "responses": ["A list stores multiple values in order."]
        },
        {
          "tag": "thanks",
          "patterns": ["thanks", "thank you"],
          "responses": ["You're welcome!"]
        },
        {
          "tag": "greeting2",
          "patterns": ["how are you", "how are you doing"],
          "responses": ["I'm doing well!"]
        },
        {
          "tag": "meaning_hello",
          "patterns": [
              "what is hello",
              "what does hello mean",
              "define hello"
          ],
          "responses": [
              "'Hello' is a greeting people use when they meet."
          ]
        },
        {
          "tag": "python",
          "patterns": [
              "what is python",
              "tell me about python",
              "define python",
              "python meaning",
              "what is python language",
              "what is python programming"
          ],
          "responses": ["Python is a programming language."]
        },
        {
          "tag": "loop",
          "patterns": [
              "what is loop",
              "what is a loop",
              "define loop",
              "tell me about loops",
              "how do loops work",
              "what is loop in python"
          ],
          "responses": ["A loop repeats code."]
        },
        {
          "tag": "function",
          "patterns": [
              "what is function",
              "what is a function",
              "what are functions",
              "define function",
              "define functions",
              "tell me about functions",
              "explain function"
          ],
          "responses": [
              "A function is reusable code."
          ]
        }
    ]
}

In [21]:
data = intents

In [33]:
def train_chatbot():
    import json
    import nltk
    from sklearn.feature_extraction.text import CountVectorizer
    from sklearn.linear_model import LogisticRegression

    global data, vectorizer, model

    with open("intents.json", "w") as file:
        json.dump(intents, file, indent=4)

    with open("intents.json", "r") as file:
        data = json.load(file)

    patterns = []
    tags = []

    for intent in data["intents"]:
        for pattern in intent["patterns"]:
            patterns.append(pattern)
            tags.append(intent["tag"])

    sentences = []

    for sentence in patterns:
        words = nltk.word_tokenize(sentence.lower())
        sentences.append(" ".join(words))

    vectorizer = CountVectorizer()
    X = vectorizer.fit_transform(sentences)

    model = LogisticRegression()
    model.fit(X, tags)

    print("Chatbot trained!")

In [34]:
train_chatbot()

Chatbot trained!


In [25]:
!pip install spacy
!python -m spacy download en_core_web_md
!pip install gradio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 20.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [26]:
import nltk
import spacy
from nltk.corpus import wordnet

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

nlp = spacy.load("en_core_web_md")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [19]:
import pickle

with open("chatbot_model.pkl", "wb") as file:
    pickle.dump(model, file)

with open("vectorizer.pkl", "wb") as file:
    pickle.dump(vectorizer, file)

print("Model saved!")

Model saved!


In [20]:
import pickle

with open("chatbot_model.pkl", "rb") as file:
    model = pickle.load(file)

with open("vectorizer.pkl", "rb") as file:
    vectorizer = pickle.load(file)

print("Model loaded!")

Model loaded!


In [43]:
import gradio as gr

def chat(message, history):
    # If the user says something like "and what about loops"
    # combine it with the previous question
    if history:
        previous_message = history[-1][0]

        if message.lower().startswith("and "):
            message = previous_message + " " + message

    return chatbot_response(message)

demo = gr.ChatInterface(
    fn=chat,
    title="My NLP Chatbot",
    description="A chatbot made using Python, NLP, spaCy and WordNet."
)

demo.launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cac90d68aa39b16565.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [17]:
def start_chat():
    print("Chatbot is ready! Type 'quit' to stop.")

    while True:
        message = input("You: ")

        if message.lower() == "quit":
            print("Bot: Goodbye!")
            break

        response = chatbot_response(message)
        print("Bot:", response)

In [18]:
start_chat()

Chatbot is ready! Type 'quit' to stop.
You: what is python 
Bot: Python is a programming language.
You: what is python in nature
Bot: In nature, python means: large Old World boas
You: what is python in biology
Bot: In biology, python means: large Old World boas
You: is python a snake
Bot: Python is a programming language.
You: is python a snake ?
Bot: Python is a programming language.
You: quit
Bot: Goodbye!
